# Reproduce the research in the attached reference manuscript, Target Fishing and Molecular Docking of Terpenes. Recapitulate the 23-terpene, four-tool target-fishing consensus design and five-target docking panel (CB2/6KPF, androgen receptor/3V49, CYP19A1/3EQM, estrogen receptor beta/1L2J, and 11-bet

Terpedia reproducibility notebook for study `study_be6fa4a4-c9eb-45e6-916e-aec999992e4f`. This notebook is generated from the immutable research snapshot `8725fda7a8c61057a326afb6c6944068f64d1bc9835b25b5ea2168fbed693908` and is compatible with Google Colab.


## Reproducibility contract

The embedded study snapshot is authoritative. Measurements, predictions, failures, and missing results remain distinct. Re-running this notebook verifies recorded provenance and reconstructs analysis tables; external scientific software or private artifacts may require the pinned environments and credentials named in the study record.


In [ ]:
import base64, hashlib, json
from pathlib import Path

EXPECTED_RESEARCH_SHA256 = "8725fda7a8c61057a326afb6c6944068f64d1bc9835b25b5ea2168fbed693908"
STUDY = json.loads(base64.b64decode("..."))
print(STUDY["study"]["title"])
print("Study:", STUDY["study"]["id"])
print("Research snapshot:", EXPECTED_RESEARCH_SHA256)


In [ ]:
def canonical(value):
    if isinstance(value, dict):
        return "{" + ",".join(json.dumps(str(k), ensure_ascii=False) + ":" + canonical(value[k]) for k in sorted(value)) + "}"
    if isinstance(value, list):
        return "[" + ",".join(canonical(item) for item in value) + "]"
    return json.dumps(value, ensure_ascii=False, separators=(",", ":"))

actual = hashlib.sha256(canonical(STUDY).encode("utf-8")).hexdigest()
assert actual == EXPECTED_RESEARCH_SHA256, (actual, EXPECTED_RESEARCH_SHA256)
print("Research snapshot integrity verified")


## Environment


In [ ]:
import sys, platform
print("Python", sys.version)
print("Platform", platform.platform())
for protocol in STUDY["protocols"]:
    for step in protocol.get("steps", []):
        print(f'{step.get("id")}: {step.get("operation")} | parameters={step.get("parameters", {})}')


## Terpedia natural-language API

The client requests safe JSON-serialized Python data. In Colab, store either a personal JWT as `TERPEDIA_TOKEN` or an approved service key as `TERPEDIA_API_KEY` in Secrets. Credentials are read at runtime and are never written into the notebook.


In [ ]:
import os, requests
from dataclasses import dataclass

try:
    from google.colab import userdata as colab_userdata
except ImportError:
    colab_userdata = None

def _secret(name):
    if colab_userdata is not None:
        try:
            return colab_userdata.get(name)
        except Exception:
            pass
    return os.environ.get(name)

@dataclass
class TerpediaResult:
    raw: dict

    def __post_init__(self):
        self.variables = self.raw.get("python_data", {}).get("variables", {})
        for name, value in self.variables.items():
            if name.isidentifier() and not hasattr(self, name):
                setattr(self, name, value)

    def frame(self, name="data"):
        value = self.variables.get(name)
        if value is None:
            return pd.DataFrame()
        if isinstance(value, list):
            return pd.json_normalize(value, sep=".")
        if isinstance(value, dict):
            return pd.json_normalize([value], sep=".")
        return pd.DataFrame({name: [value]})

class TerpediaClient:
    def __init__(self, base_url="https://api.terpedia.com", token=None, api_key=None):
        self.base_url = base_url.rstrip("/")
        self.token = token or _secret("TERPEDIA_TOKEN")
        self.api_key = api_key or _secret("TERPEDIA_API_KEY")

    def query(self, question, **options):
        headers = {"Content-Type": "application/json"}
        if self.token:
            headers["Authorization"] = f"Bearer {self.token}"
        elif self.api_key:
            headers["X-API-Key"] = self.api_key
        payload = {"query": question, "response_format": "python_data", **options}
        response = requests.post(f"{self.base_url}/api/v1/query", headers=headers, json=payload, timeout=180)
        response.raise_for_status()
        return TerpediaResult(response.json())

terpedia = TerpediaClient()
research_question = next((p.get("objective") for p in reversed(STUDY["protocols"]) if p.get("objective")), STUDY["study"].get("hypothesis") or STUDY["study"]["title"] )
print("research_question =", research_question)
print("Ready: terpedia_response = terpedia.query(research_question)")
print("Then use terpedia_response.answer, .data, .citations, .variables, or .frame().")


## Study records


In [ ]:
import pandas as pd

runs = pd.json_normalize(STUDY["runs"], sep=".")
evidence = pd.json_normalize(STUDY["evidence"], sep=".")
artifacts = pd.json_normalize(STUDY["artifacts"], sep=".")
display(runs if not runs.empty else pd.DataFrame({"status": ["No runs recorded"]}))
display(evidence if not evidence.empty else pd.DataFrame({"status": ["No evidence recorded"]}))
display(artifacts if not artifacts.empty else pd.DataFrame({"status": ["No artifacts recorded"]}))


## Provenance and artifact verification


In [ ]:
from urllib.request import urlopen

def verify_https_artifact(record):
    uri = record.get("uri", "")
    expected = record.get("sha256")
    if not uri.startswith("https://"):
        return {"uri": uri, "status": "credential_or_protocol_required", "expected_sha256": expected}
    with urlopen(uri) as response:
        actual = hashlib.sha256(response.read()).hexdigest()
    return {"uri": uri, "status": "verified" if actual == expected else "checksum_mismatch", "expected_sha256": expected, "actual_sha256": actual}

artifact_manifest = [{k: item.get(k) for k in ("id", "name", "uri", "sha256", "role")} for item in STUDY["artifacts"]]
display(pd.DataFrame(artifact_manifest))
print("Call verify_https_artifact(record) for public HTTPS artifacts. gs:// artifacts require authenticated Google Cloud access.")


## Recorded results


In [ ]:
numeric_rows = []
for item in STUDY["evidence"]:
    value = item.get("value")
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        numeric_rows.append({
            "subject": item.get("subject", {}).get("label") or item.get("subject", {}).get("id"),
            "predicate": item.get("predicate"), "value": value, "unit": item.get("unit"),
            "kind": item.get("kind"), "status": item.get("evidenceStatus"), "sha256": item.get("sha256")
        })
numeric_evidence = pd.DataFrame(numeric_rows)
display(numeric_evidence if not numeric_evidence.empty else pd.DataFrame({"status": ["No scalar numeric evidence recorded"]}))


In [ ]:
import matplotlib.pyplot as plt
if not runs.empty and "status" in runs:
    runs["status"].value_counts().plot(kind="bar", title="Recorded run outcomes")
    plt.xlabel("Status")
    plt.ylabel("Runs")
    plt.tight_layout()
    plt.show()
if not numeric_evidence.empty:
    numeric_evidence.groupby("predicate")["value"].count().sort_values(ascending=False).head(20).plot(kind="bar", title="Numeric evidence by predicate")
    plt.ylabel("Records")
    plt.tight_layout()
    plt.show()


## Reference-paper study reproduction

These cells are generated by Terpedia from the recorded `study-replication` tool result. They reconstruct the author-supplied target-fishing analysis, reference MOE analysis, Terpedia Vina comparison, size effects, ligand efficiency, and TF+/TF- enrichment from frozen row-level datasets.


In [ ]:
replication_runs = [run for run in STUDY["runs"] if run.get("outputs", {}).get("report", {}).get("analysis", {}).get("datasets")]
assert replication_runs, "No recorded Terpedia study-replication result"
REPLICATION = replication_runs[-1]["outputs"]["report"]
DATASETS = REPLICATION["analysis"]["datasets"]

target_predictions = pd.DataFrame(DATASETS["target_predictions"])
author_consensus = pd.DataFrame(DATASETS["author_consensus"])
target_frequencies = pd.DataFrame(DATASETS["target_frequencies"])
vina_docking = pd.DataFrame(DATASETS["docking_observations"])
moe_docking = pd.DataFrame(DATASETS["reference_docking_observations"])

assert len(target_predictions) == 1546
assert author_consensus[["terpene", "target_name"]].drop_duplicates().shape[0] == 257
assert len(vina_docking) == 130 and len(moe_docking) == 129
print({"predictions": len(target_predictions), "author_consensus_rows": len(author_consensus), "vina": len(vina_docking), "moe": len(moe_docking)})


### Target-fishing consensus and tool dependence


In [ ]:
import re, unicodedata

def normalized_name(value):
    value = str(value or "").replace("α", "alpha").replace("β", "beta").replace("γ", "gamma")
    value = unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode("ascii").casefold()
    return re.sub(r"[^a-z0-9]", "", value)

def stable_target_key(row):
    return row.get("chembl_id") or row.get("uniprot_id") or normalized_name(row.get("target_name"))

for frame in (target_predictions, author_consensus):
    frame["ligand_key"] = frame["terpene"].map(normalized_name)
    frame["target_key"] = frame.apply(stable_target_key, axis=1)

def consensus_pairs(frame, omitted_tool=None, minimum_tools=2):
    current = frame if omitted_tool is None else frame[frame["tool"] != omitted_tool]
    grouped = current.groupby(["ligand_key", "target_key"], as_index=False)["tool"].agg(lambda values: sorted(set(values))).rename(columns={"tool": "tools"})
    grouped["tool_count"] = grouped["tools"].map(len)
    return grouped[grouped["tool_count"] >= minimum_tools].reset_index(drop=True)

recomputed_consensus = consensus_pairs(target_predictions)
author_pairs = set(map(tuple, author_consensus[["ligand_key", "target_key"]].drop_duplicates().to_numpy()))
recomputed_pairs = set(map(tuple, recomputed_consensus[["ligand_key", "target_key"]].to_numpy()))
consensus_comparison = pd.Series({
    "recomputed_pairs": len(recomputed_pairs), "author_pairs": len(author_pairs),
    "overlap": len(recomputed_pairs & author_pairs),
    "precision_vs_author": len(recomputed_pairs & author_pairs) / len(recomputed_pairs),
    "recall_vs_author": len(recomputed_pairs & author_pairs) / len(author_pairs),
}, name="value").to_frame()
display(consensus_comparison)

tools = ["SwissTargetPrediction", "SEA", "STarFish", "PPB2"]
ablation = []
for omitted in tools:
    reduced = consensus_pairs(target_predictions, omitted_tool=omitted)
    pairs = set(map(tuple, reduced[["ligand_key", "target_key"]].to_numpy()))
    ablation.append({"omitted_tool": omitted, "consensus_pairs": len(pairs), "pairs_lost": len(recomputed_pairs - pairs), "author_recall": len(pairs & author_pairs) / len(author_pairs)})
leave_one_tool_out = pd.DataFrame(ablation).sort_values("consensus_pairs", ascending=False)
display(leave_one_tool_out)
display(target_frequencies[target_frequencies["selected_top_five"]][["rank", "target_name", "terpenes_predicting"]])


### TF labels, ligand efficiency, and molecular-size effects


In [ ]:
TARGET_ALIASES = {
    "cannabinoidreceptor2": "CB2", "cannabinoidcb2receptor": "CB2", "cb2": "CB2",
    "androgenreceptor": "AR", "ar": "AR", "cytochromep45019a1": "CYP19A1", "aromatase": "CYP19A1", "cyp19a1": "CYP19A1",
    "estrogenreceptorbeta": "ERbeta", "erbeta": "ERbeta", "11betahydroxysteroiddehydrogenase1": "11beta-HSD1", "11betahsd1": "11beta-HSD1", "bhd": "11beta-HSD1",
}
def target_key(value):
    key = normalized_name(value)
    return TARGET_ALIASES.get(key, key)

author_tf_pairs = {(normalized_name(row.terpene), target_key(row.target_name)) for row in author_consensus.itertuples()}
def decorate_docking(frame, method):
    result = frame.copy()
    result["method"] = method
    result["tf_positive"] = [
        (normalized_name(row.ligand), target_key(row.target)) in author_tf_pairs for row in result.itertuples()
    ]
    result["ligand_efficiency"] = result["score_kcal_mol"].abs() / result["heavy_atoms"]
    result["score_magnitude"] = result["score_kcal_mol"].abs()
    return result

docking = pd.concat([decorate_docking(moe_docking, "MOE GBVI/WSA dG"), decorate_docking(vina_docking, "AutoDock Vina")], ignore_index=True)

correlation_rows = []
for (method, target), group in docking.groupby(["method", "target"]):
    known = group[group["ligand_class"] == "known_ligand"]
    for label, ligand_class in (("monoterpenes", "monoterpene"), ("sesquiterpenes", "sesquiterpene")):
        terpene_class = group[group["ligand_class"] == ligand_class]
        correlation_rows.append({
            "method": method, "target": target, "series": label,
            "terpenes_plus_known": pd.concat([terpene_class, known])[["heavy_atoms", "score_magnitude"]].corr().iloc[0, 1],
            "terpenes_only": terpene_class[["heavy_atoms", "score_magnitude"]].corr().iloc[0, 1],
        })
size_correlations = pd.DataFrame(correlation_rows)
display(size_correlations)

ligand_efficiency_summary = docking.groupby(["method", "target", "ligand_class"])["ligand_efficiency"].agg(["count", "mean", "median", "min", "max"]).reset_index()
display(ligand_efficiency_summary)


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 5, figsize=(20, 7), sharex=False, sharey=False)
targets = ["CB2", "AR", "11beta-HSD1", "CYP19A1", "ERbeta"]
colors = {"known_ligand": "#b33b32", "monoterpene": "#28735d", "sesquiterpene": "#d28b27"}
for row_index, method in enumerate(["MOE GBVI/WSA dG", "AutoDock Vina"]):
    for column_index, target in enumerate(targets):
        ax = axes[row_index, column_index]
        subset = docking[(docking["method"] == method) & (docking["target"] == target)]
        for ligand_class, points in subset.groupby("ligand_class"):
            ax.scatter(points["heavy_atoms"], points["score_magnitude"], label=ligand_class, color=colors[ligand_class], alpha=.8)
        ax.set_title(f"{target} | {method}", fontsize=9)
        ax.set_xlabel("Heavy atoms")
        if column_index == 0: ax.set_ylabel("|Docking score|")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3)
fig.tight_layout(rect=[0, 0, 1, .94])
plt.show()


### Target-fishing enrichment by docking rank

As in the reference paper, known ligands and TF-positive terpenes are counted as positives. Undefined one-class cases remain explicitly undefined. Values are descriptive because each target contains only 23 terpenes.


In [ ]:
import math
def enrichment_curve(population):
    population = population.sort_values("score_kcal_mol").reset_index(drop=True)
    positive = (population["ligand_class"] == "known_ligand") | population["tf_positive"]
    identifiable = 0 < int(positive.sum()) < len(population)
    rows = []
    for percent in range(10, 101, 10):
        top_n = max(1, math.ceil(len(population) * percent / 100))
        factor = (positive.iloc[:top_n].mean() / positive.mean()) if identifiable else None
        rows.append({"percent": percent, "top_n": top_n, "enrichment_factor": factor, "identifiable": identifiable})
    return rows

enrichment_rows = []
for (method, target), group in docking.groupby(["method", "target"]):
    known = group[group["ligand_class"] == "known_ligand"]
    for label, ligand_class in (("monoterpenes", "monoterpene"), ("sesquiterpenes", "sesquiterpene")):
        population = pd.concat([known, group[group["ligand_class"] == ligand_class]], ignore_index=True)
        for row in enrichment_curve(population):
            enrichment_rows.append({"method": method, "target": target, "series": label, **row})
enrichment = pd.DataFrame(enrichment_rows)
display(enrichment)

fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
for row_index, method in enumerate(["MOE GBVI/WSA dG", "AutoDock Vina"]):
    for column_index, series in enumerate(["monoterpenes", "sesquiterpenes"]):
        ax = axes[row_index, column_index]
        subset = enrichment[(enrichment["method"] == method) & (enrichment["series"] == series)]
        for target, points in subset.groupby("target"):
            ax.plot(points["percent"], points["enrichment_factor"], marker="o", label=target)
        ax.axhline(1, color="#666", linestyle="--", linewidth=1)
        ax.set_title(f"{method} | {series}")
        ax.set_ylabel("Enrichment factor")
        ax.set_xlabel("Top-ranked fraction (%)")
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
plt.show()


### MOE-Vina rank concordance


In [ ]:
concordance_rows = []
for target in sorted(set(moe_docking["target"]) & set(vina_docking["target"])):
    shared = moe_docking[moe_docking["target"] == target][["ligand", "score_kcal_mol"]].merge(
        vina_docking[vina_docking["target"] == target][["ligand", "score_kcal_mol"]], on="ligand", suffixes=("_moe", "_vina")
    )
    moe_top = set(shared.nsmallest(5, "score_kcal_mol_moe")["ligand"])
    vina_top = set(shared.nsmallest(5, "score_kcal_mol_vina")["ligand"])
    concordance_rows.append({
        "target": target, "shared_ligands": len(shared),
        "spearman_rho": shared[["score_kcal_mol_moe", "score_kcal_mol_vina"]].corr(method="spearman").iloc[0, 1],
        "top_5_overlap": len(moe_top & vina_top), "top_5_jaccard": len(moe_top & vina_top) / len(moe_top | vina_top),
    })
method_concordance = pd.DataFrame(concordance_rows)
display(method_concordance)


### Export reproduced analyses


In [ ]:
analysis_output = Path("/content/terpedia-reference-reproduction" if "google.colab" in sys.modules else "terpedia-reference-reproduction")
analysis_output.mkdir(parents=True, exist_ok=True)
target_predictions.to_csv(analysis_output / "target-fishing-predictions.csv", index=False)
author_consensus.to_csv(analysis_output / "author-consensus.csv", index=False)
leave_one_tool_out.to_csv(analysis_output / "leave-one-tool-out.csv", index=False)
docking.to_csv(analysis_output / "moe-vina-docking.csv", index=False)
size_correlations.to_csv(analysis_output / "size-score-correlations.csv", index=False)
ligand_efficiency_summary.to_csv(analysis_output / "ligand-efficiency-summary.csv", index=False)
enrichment.to_csv(analysis_output / "tf-docking-enrichment.csv", index=False)
method_concordance.to_csv(analysis_output / "moe-vina-concordance.csv", index=False)
print("Wrote", len(list(analysis_output.iterdir())), "Terpedia-generated analysis artifacts to", analysis_output)


## Export


In [ ]:
summary = {
    "study_id": STUDY["study"]["id"],
    "research_snapshot_sha256": EXPECTED_RESEARCH_SHA256,
    "protocols": len(STUDY["protocols"]), "runs": len(STUDY["runs"]),
    "evidence_records": len(STUDY["evidence"]), "artifacts": len(STUDY["artifacts"]),
    "run_statuses": runs["status"].value_counts().to_dict() if not runs.empty and "status" in runs else {}
}
Path("terpedia-reproducibility-summary.json").write_text(json.dumps(summary, indent=2))
summary
